# Engine v4 · Stockfish

Stockfish 16 carries two evaluations: the NNUE network (the default) and the hand-written classical one, and the UCI option `Use NNUE` switches between them. Every Stockfish move below prints what its search reports: the depth it reached, the nodes it searched, the nodes per second and the score.

Stockfish is always White and searches to a fixed depth (8 by default), so that NNUE on and off build trees of the same depth; set `SECONDS` to give it a clock instead. It needs Stockfish 12–16: Stockfish 16.1 and later removed the classical evaluation and the `Use NNUE` option with it.

Set the opponent in the settings cell: `"you"`, `"sunfish"`, `"classical"` (the same Stockfish with NNUE off, same depth), `"stockfish"` (a rating-limited Stockfish at `ELO`) or `"random"`.



In [ ]:
%pip install chess -q
OPPONENT  = "sunfish"     # "you" | "sunfish" | "classical" | "stockfish" | "random"
NNUE      = True          # White's evaluation: True = the network, False = the hand-written one
DEPTH     = 8             # Stockfish's fixed depth
SECONDS   = None          # or seconds a move instead of a depth, e.g. 0.1
ELO       = 1350          # rating limit for OPPONENT = "stockfish"
MOVES     = ""            # Black's moves to replay, in SAN, e.g. "e5 Nc6 Nf6" (OPPONENT = "you")
SUNFISH   = "sunfish/sunfish.py"
MAX_MOVES = 150

import os
if not os.path.isdir("scripts") and os.path.isdir(os.path.join("..", "scripts")):
    os.chdir("..")                      # opened from notebooks/ -- work from the portfolio root instead
print("working in", os.path.basename(os.getcwd()))


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
working in content


## The pieces: finding Stockfish, opening it with NNUE on or off, a keyboard mover, the search report


In [ ]:
import datetime
import os
import random
import shutil
import sys
import time

import chess
import chess.engine
import chess.pgn


def find_stockfish():
    for p in [os.environ.get("STOCKFISH_PATH"), shutil.which("stockfish"), "/usr/games/stockfish",
              "/usr/local/bin/stockfish", "/opt/homebrew/bin/stockfish", "./stockfish", "./stockfish.exe"]:
        if p and os.path.isfile(p):
            return p
    return None


def open_stockfish(path, nnue=True):
    eng = chess.engine.SimpleEngine.popen_uci(path)
    if "Use NNUE" not in eng.options:
        eng.quit()
        sys.exit(f"{eng.id.get('name', 'this Stockfish')} has no 'Use NNUE' option -- use Stockfish 12-16 "
                 "(github.com/official-stockfish/Stockfish/releases/tag/sf_16) and set STOCKFISH_PATH")
    eng.configure({"Threads": 1, "Hash": 64, "Use NNUE": nnue})
    return eng


def human_mover(scripted):
    queue = list(scripted)

    def move(board):
        while True:
            try:
                text = queue.pop(0) if queue else input(f"  your move ({board.fullmove_number}...) in SAN, or 'resign': ").strip()
            except (EOFError, NotImplementedError):            # no keyboard here: stop the game, keep the PGN
                raise KeyboardInterrupt
            if text.lower() in ("resign", "quit", "stop"):
                raise KeyboardInterrupt
            try:
                return board.parse_san(text)
            except ValueError:
                print(f"  {text!r} is not legal here -- try again")
    return move


def report(info):
    """One line of what a search said: depth, nodes, nodes per second, score from White's side."""
    score = info.get("score")
    nodes, nps = info.get("nodes", 0), info.get("nps", 0)
    text = f"  depth {info.get('depth', '?'):>2}  {nodes:>9,} nodes  {nps:>9,} n/s"
    if score is not None:
        w = score.white()
        text += f"  mate {w.mate():+d}" if w.is_mate() else f"  score {w.score():+d}"
    return text, nodes or 0


## The game loop — `play_stockfish()`



In [ ]:
def play_stockfish(opponent="you", nnue=True, depth=8, seconds=None, elo=1350, moves="",
                   sunfish=os.path.join("sunfish", "sunfish.py"), max_moves=150):
    """Stockfish (White, NNUE on or off) against opponent -- 'you', 'sunfish', 'classical', 'stockfish' or 'random'.
    Prints every move with Stockfish's depth, nodes, nodes per second and score, then the final board; writes
    games/v5-stockfish-<nnue|classical>-vs-<opponent>.pgn. Returns the board, and the nodes and seconds Stockfish spent."""
    path = find_stockfish()
    if not path:
        raise SystemExit("Stockfish not found -- set STOCKFISH_PATH or put it on your PATH")
    engine = open_stockfish(path, nnue)
    limit = chess.engine.Limit(time=seconds) if seconds else chess.engine.Limit(depth=depth)
    setting = f"{seconds:g} s a move" if seconds else f"depth {depth}"
    white_name = f"{engine.id.get('name', 'Stockfish')} ({'NNUE' if nnue else 'classical'}, {setting})"
    print("engine.id =", engine.id, "| network:", engine.options["EvalFile"].default)

    opp = None
    if opponent == "classical":
        opp = open_stockfish(path, nnue=False)
        opponent_move = lambda b: opp.play(b, limit, info=chess.engine.INFO_ALL)
        opponent_name = f"{opp.id.get('name', 'Stockfish')} (classical, {setting})"
    elif opponent == "stockfish":
        opp = open_stockfish(path, nnue=True)
        o = opp.options["UCI_Elo"]
        elo = max(o.min, min(o.max, elo))
        opp.configure({"UCI_LimitStrength": True, "UCI_Elo": elo})
        opponent_move = lambda b: opp.play(b, chess.engine.Limit(time=0.1))
        opponent_name = f"Stockfish ({elo})"
    elif opponent == "sunfish":
        if not os.path.isfile(sunfish):
            raise SystemExit(f"{sunfish} not found -- git clone https://github.com/thomasahle/sunfish.git")
        folder = os.path.dirname(os.path.abspath(sunfish))
        opp = chess.engine.SimpleEngine.popen_uci([sys.executable, os.path.basename(sunfish)], cwd=folder)
        opponent_move = lambda b: opp.play(b, chess.engine.Limit(time=0.2), info=chess.engine.INFO_ALL)
        opponent_name = "Sunfish (0.2 s a move)"
    elif opponent == "random":
        random.seed(1)
        opponent_move, opponent_name = (lambda b: random.choice(list(b.legal_moves))), "random mover"
    else:
        opponent_move, opponent_name = human_mover(moves.split()), "me"
    print(f"White: {white_name}    Black: {opponent_name}\n")

    board, t_start, total_nodes, stopped = chess.Board(), time.time(), 0, None
    sf_seconds = 0.0                                       # Stockfish's own thinking time, for nodes per second
    try:
        while not board.is_game_over(claim_draw=True) and board.fullmove_number <= max_moves:
            t0 = time.time()
            if board.turn == chess.WHITE:
                result = engine.play(board, limit, info=chess.engine.INFO_ALL)
                move, who = result.move, "SF "
                note, nodes = report(result.info)
                total_nodes += nodes
                sf_seconds += time.time() - t0
            else:
                answer = opponent_move(board)
                if isinstance(answer, chess.engine.PlayResult):
                    move, note = answer.move, report(answer.info)[0] if answer.info else ""
                else:
                    move, note = answer, ""
                who = "opp"
            num = f"{board.fullmove_number}." if board.turn == chess.WHITE else "   "
            san = board.san(move)
            board.push(move)
            print(f"  {num:>4} {who} {san:<8} ({time.time() - t0:4.1f}s){note}")
    except KeyboardInterrupt:
        stopped = "stopped by hand"
    finally:
        engine.quit()
        if opp:
            opp.quit()

    over = board.is_game_over(claim_draw=True)
    outcome = board.outcome(claim_draw=True)
    print(f"\n{len(board.move_stack)} plies in {time.time() - t_start:.0f}s.  Result "
          f"{board.result(claim_draw=True) if over else '*'}  "
          f"({outcome.termination.name.lower() if outcome else 'unfinished'})   "
          f"Stockfish searched {total_nodes:,} nodes in all.")
    print(board)

    game = chess.pgn.Game.from_board(board)
    game.headers.update({"Event": "Gambit Lab · Engine v5", "Site": "Toledo, OH",
                         "Date": datetime.date.today().strftime("%Y.%m.%d"), "Round": "-",
                         "White": white_name, "Black": opponent_name,
                         "Result": board.result(claim_draw=True) if over else "*"})
    if stopped:
        game.headers["Termination"] = stopped
    os.makedirs("games", exist_ok=True)
    who = "me" if opponent == "you" else opponent
    out = os.path.join("games", f"v5-stockfish-{'nnue' if nnue else 'classical'}-vs-{who}.pgn")
    with open(out, "w") as f:
        print(game, file=f, end="\n\n")
    print("saved game ->", out)
    return board, total_nodes, sf_seconds


## Play

Against you: set `OPPONENT = "you"` and answer the prompt in SAN. Against the others: as set. The PGN lands in `games/`.

In [ ]:
!apt-get update && apt-get install -y stockfish
!git clone https://github.com/thomasahle/sunfish.git

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
stockfish is already the newest version (16-1build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Cloning into 'sunfish'...
remote: Enumerating object

In [ ]:
board, nodes, secs = play_stockfish('You', NNUE, DEPTH, SECONDS, ELO, MOVES, SUNFISH, MAX_MOVES)


engine.id = {'name': 'Stockfish 16', 'author': 'the Stockfish developers (see AUTHORS file)'} | network: nn-5af11540bbfe.nnue
White: Stockfish 16 (NNUE, depth 8)    Black: me

    1. SF  e4       ( 0.0s)  depth  8      2,105 nodes    233,888 n/s  score +40
  your move (1...) in SAN, or 'resign': 'resign'
  "'resign'" is not legal here -- try again
  your move (1...) in SAN, or 'resign': resign

1 plies in 18s.  Result *  (unfinished)   Stockfish searched 2,105 nodes in all.
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R
saved game -> games/v5-stockfish-nnue-vs-You.pgn


## The switch





## The switch

The same opponent again at the same depth, with White's network switched the other way. Then read the two games side by side: the nodes per second (the network is slower per position), the nodes per move (it often needs fewer for the same depth) and the result. The second PGN is saved under its own name.

In [ ]:
first = (NNUE, board, nodes, secs)
board2, nodes2, secs2 = play_stockfish(OPPONENT, not NNUE, DEPTH, SECONDS, ELO, MOVES, SUNFISH, MAX_MOVES)
second = (not NNUE, board2, nodes2, secs2)

print(f"\n  {'White':<10} {'result':<8} {'SF moves':>8} {'nodes/move':>11} {'nodes/s':>10}")
for on, b, n, s in sorted([first, second], key=lambda r: not r[0]):
    k = (len(b.move_stack) + 1) // 2                       # Stockfish's moves: it is White and moves first
    result = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "*"
    print(f"  {'NNUE' if on else 'classical':<10} {result:<8} {k:>8} {n / max(1, k):>11,.0f} {n / max(s, 1e-9):>10,.0f}")


engine.id = {'name': 'Stockfish 16', 'author': 'the Stockfish developers (see AUTHORS file)'} | network: nn-5af11540bbfe.nnue
White: Stockfish 16 (classical, depth 8)    Black: Sunfish (0.2 s a move)

    1. SF  Nc3      ( 0.0s)  depth  8     12,725 nodes    848,333 n/s  score +4
       opp e5       ( 0.4s)  depth  3      1,315 nodes     11,701 n/s  score -6
    2. SF  d4       ( 0.0s)  depth  8      6,401 nodes    711,222 n/s  score +33
       opp exd4     ( 0.4s)  depth  3        494 nodes     12,989 n/s  score -7
    3. SF  Qxd4     ( 0.0s)  depth  8        895 nodes    447,500 n/s  score +33
       opp Nc6      ( 0.3s)  depth  4      3,292 nodes      9,781 n/s  score -3
    4. SF  Qe3+     ( 0.0s)  depth  8      1,166 nodes    388,666 n/s  score +31
       opp Be7      ( 0.2s)  depth  3      1,186 nodes      6,667 n/s  score -22
    5. SF  Nd5      ( 0.0s)  depth  8      7,805 nodes    487,812 n/s  score +53
       opp Nf6      ( 0.2s)  depth  4      1,654 nodes      9,358 n/s  sco